13.Experiment 1 - Corpus Stastitic

In [5]:

### **AI assistance statement**

#Tool:** Gemini (Google)
#Purpose:** Hỗ trợ viết đoạn mã Python mẫu trong Jupyter Notebook để xử lý, thống kê dữ liệu corpus (`c4-train.00000-of-01024-30K.json`) và vẽ biểu đồ phân phối tần suất N-gram cho Experiment 1.
#What was generated:** Đoạn mã Python hoàn chỉnh thực hiện các tác vụ:
#1. Đọc dữ liệu JSON và làm sạch/phân đoạn câu/tách từ (tokenization).
#2. Thống kê số lượng documents, sentences, tokens, vocabulary size.
#3. Đếm số lượng unique n-grams, singletons (n-gram xuất hiện 1 lần) cho Unigram, Bigram, Trigram.
#4. Trực quan hóa Top 20 N-grams bằng thư viện `matplotlib`.


#What was modified:** Tinh chỉnh các đường dẫn file dữ liệu cho đúng với tên cấu trúc thư mục thực tế (`c4-train.00000-of-01024-30K.json`), điều chỉnh thông số vẽ biểu đồ (`figsize`, `rotation` của nhãn trục X) để giao diện hiển thị rõ ràng và đẹp mắt hơn trong Jupyter Notebook.
#How the result was verified:** Khởi chạy trực tiếp đoạn code trong môi trường Jupyter Notebook trên máy cá nhân, kiểm tra đầu ra của bảng thống kê xem các thông số có logic không (đảm bảo số lượng Unique Trigram > Bigram > Unigram) và kiểm tra biểu đồ đã hiển thị đúng tần suất của các từ/cụm từ phổ biến.
import json
import re
from collections import Counter
import matplotlib.pyplot as plt
import pandas as pd

json_path = 'c4-train.00000-of-01024-30K.json'

documents = []
with open(json_path, 'r', encoding='utf-8') as f:
    for line in f:
        data = json.loads(line)
        documents.append(data['text'])

num_documents = len(documents)

def tokenize(text):
    # Chuyển chữ thường và tách từ theo từ (word boundaries)
    return re.findall(r'\b\w+\b', text.lower())

sentences = []
for doc in documents:
    raw_sents = re.split(r'[.!?]+', doc)
    for s in raw_sents:
        tokens = tokenize(s)
        if tokens:
            sentences.append(tokens)

num_sentences = len(sentences)

all_tokens = [token for sent in sentences for token in sent]
num_tokens = len(all_tokens)
vocab_size = len(set(all_tokens))

def get_ngrams_counts(sentences, n):
    counts = Counter()
    for sent in sentences:
        if len(sent) >= n:
            for i in range(len(sent) - n + 1):
                ngram = tuple(sent[i:i+n])
                counts[ngram] += 1
    return counts

unigram_counts = Counter(all_tokens)
bigram_counts = get_ngrams_counts(sentences, 2)
trigram_counts = get_ngrams_counts(sentences, 3)

num_unigrams = len(unigram_counts)
num_bigrams = len(bigram_counts)
num_trigrams = len(trigram_counts)

single_unigrams = sum(1 for count in unigram_counts.values() if count == 1)
single_bigrams = sum(1 for count in bigram_counts.values() if count == 1)
single_trigrams = sum(1 for count in trigram_counts.values() if count == 1)

stats_df = pd.DataFrame({
    'Metric': ['# Documents', '# Sentences', '# Tokens', 'Vocabulary Size'],
    'Value': [num_documents, num_sentences, num_tokens, vocab_size]
})
print("=== CORPUS OVERVIEW ===")
print(stats_df.to_string(index=False))

ngram_df = pd.DataFrame({
    'Model': ['Unigram', 'Bigram', 'Trigram'],
    '# Unique N-grams': [num_unigrams, num_bigrams, num_trigrams],
    '# Singletons (Count = 1)': [single_unigrams, single_bigrams, single_trigrams],
    '% Singletons': [
        f"{(single_unigrams/num_unigrams)*100:.2f}%",
        f"{(single_bigrams/num_bigrams)*100:.2f}%",
        f"{(single_trigrams/num_trigrams)*100:.2f}%"
    ]
})
print("\n=== N-GRAM STATISTICS ===")
print(ngram_df.to_string(index=False))

=== CORPUS OVERVIEW ===
         Metric    Value
    # Documents    30000
    # Sentences   686005
       # Tokens 11090891
Vocabulary Size   193837

=== N-GRAM STATISTICS ===
  Model  # Unique N-grams  # Singletons (Count = 1) % Singletons
Unigram            193837                     94219       48.61%
 Bigram           2746349                   1961404       71.42%
Trigram           6407002                   5525379       86.24%


In [6]:
import numpy as np
import pandas as pd
from ngram_lm import NGramLanguageModel

# 1. Chia tập dữ liệu Train / Validation / Test (80% / 10% / 10%)
# Giả sử 'sentences' đã được tạo từ Experiment 1
np.random.seed(42)
shuffled_sentences = sentences.copy()
np.random.shuffle(shuffled_sentences)

n_total = len(shuffled_sentences)
n_train = int(n_total * 0.8)
n_val = int(n_total * 0.1)

train_sentences = shuffled_sentences[:n_train]
val_sentences = shuffled_sentences[n_train:n_train + n_val]
test_sentences = shuffled_sentences[n_train + n_val:]

print(f"Train size: {len(train_sentences)} sentences")
print(f"Validation size: {len(val_sentences)} sentences")
print(f"Test size: {len(test_sentences)} sentences")

# 2. Hàm hỗ trợ tính Corpus Perplexity
def calculate_corpus_perplexity(model, eval_corpus):
    total_log_prob = 0.0
    total_ngrams = 0
    
    for sentence in eval_corpus:
        ngrams = model._get_ngrams(sentence)
        N = len(ngrams)
        log_prob_sum = model.sentence_log_probability(sentence)
        
        # Nếu dính zero probability (log_prob = -inf) trong MLE
        if log_prob_sum == float('-inf'):
            return float('inf')
        
        total_log_prob += log_prob_sum
        total_ngrams += N
        
    if total_ngrams == 0:
        return float('inf')
        
    # Công thức Perplexity toàn corpus: exp(-1/N * sum(log_prob))
    return np.exp(-total_log_prob / total_ngrams)

# 3. Khởi tạo và huấn luyện 4 mô hình
models = {
    "Bigram MLE": NGramLanguageModel(n=2, smoothing=False),
    "Bigram Laplace": NGramLanguageModel(n=2, smoothing=True),
    "Trigram MLE": NGramLanguageModel(n=3, smoothing=False),
    "Trigram Laplace": NGramLanguageModel(n=3, smoothing=True),
}

results = []

for name, model in models.items():
    print(f"Training {name}...")
    model.fit(train_sentences)
    
    train_ppl = calculate_corpus_perplexity(model, train_sentences)
    val_ppl = calculate_corpus_perplexity(model, val_sentences)
    test_ppl = calculate_corpus_perplexity(model, test_sentences)
    
    results.append({
        "Model": name,
        "Train PPL": round(train_ppl, 2) if train_ppl != float('inf') else "inf",
        "Valid PPL": round(val_ppl, 2) if val_ppl != float('inf') else "inf",
        "Test PPL": round(test_ppl, 2) if test_ppl != float('inf') else "inf"
    })

# 4. Hiển thị bảng kết quả
exp2_df = pd.DataFrame(results)
print("\n=== EXPERIMENT 2 RESULTS: MLE VS SMOOTHING ===")
print(exp2_df.to_string(index=False))

# Lưu vào results.csv
exp2_df.to_csv("results.csv", index=False)

Train size: 548804 sentences
Validation size: 68600 sentences
Test size: 68601 sentences
Training Bigram MLE...
Training Bigram Laplace...
Training Trigram MLE...
Training Trigram Laplace...

=== EXPERIMENT 2 RESULTS: MLE VS SMOOTHING ===
          Model  Train PPL Valid PPL  Test PPL
     Bigram MLE     161.66       inf       inf
 Bigram Laplace    5440.83   6607.77   6577.39
    Trigram MLE      14.64       inf       inf
Trigram Laplace   30331.83  47229.63  47123.51


19. EXPERIMENT 3

In [7]:
import numpy as np
import pandas as pd
from ngram_lm import NGramLanguageModel

# 1. Hàm tính Corpus Perplexity chuẩn xác (Sử dụng Log Prob)
def calculate_corpus_perplexity(model, eval_corpus):
    total_log_prob = 0.0
    total_ngrams = 0
    
    for sentence in eval_corpus:
        ngrams = model._get_ngrams(sentence)
        N = len(ngrams)
        log_prob_sum = model.sentence_log_probability(sentence)
        
        if log_prob_sum == float('-inf'):
            return float('inf')
        
        total_log_prob += log_prob_sum
        total_ngrams += N
        
    if total_ngrams == 0:
        return float('inf')
        
    return np.exp(-total_log_prob / total_ngrams)

# 2. Khởi tạo 3 mô hình với Laplace Smoothing
models_exp3 = {
    "Unigram": NGramLanguageModel(n=1, smoothing=True),
    "Bigram": NGramLanguageModel(n=2, smoothing=True),
    "Trigram": NGramLanguageModel(n=3, smoothing=True)
}

exp3_results = []

# 3. Huấn luyện và đánh giá Perplexity
for name, model in models_exp3.items():
    print(f"Training {name} Language Model...")
    model.fit(train_sentences)
    
    train_ppl = calculate_corpus_perplexity(model, train_sentences)
    val_ppl = calculate_corpus_perplexity(model, val_sentences)
    test_ppl = calculate_corpus_perplexity(model, test_sentences)
    
    exp3_results.append({
        "Model": name,
        "Train PPL": round(train_ppl, 2),
        "Validation PPL": round(val_ppl, 2),
        "Test PPL": round(test_ppl, 2)
    })

# 4. Hiển thị bảng kết quả Experiment 3
exp3_df = pd.DataFrame(exp3_results)
print("\n=== EXPERIMENT 3 RESULTS: PERPLEXITY EVALUATION ===")
print(exp3_df.to_string(index=False))

# Cập nhật kết quả vào results.csv
exp3_df.to_csv("results.csv", mode='a', index=False)

Training Unigram Language Model...
Training Bigram Language Model...
Training Trigram Language Model...

=== EXPERIMENT 3 RESULTS: PERPLEXITY EVALUATION ===
  Model  Train PPL  Validation PPL  Test PPL
Unigram    2061.99         2103.57   2083.36
 Bigram    5440.83         6607.77   6577.39
Trigram   30331.83        47229.63  47123.51


In [11]:
import numpy as np
from ngram_lm import NGramLanguageModel

# 1. Khởi tạo và huấn luyện Bigram Model (dùng Laplace Smoothing)
bigram_model = NGramLanguageModel(n=2, smoothing=True)
bigram_model.fit(train_sentences)

# 2. Định nghĩa hàm kiểm tra dự đoán từ tiếp theo
def inspect_next_word(model, context_str, top_k=5):
    context_tokens = context_str.lower().split()
    
    # Lấy context tuple phù hợp với n-gram model
    if model.n > 1:
        context_tuple = tuple(context_tokens[-(model.n - 1):])
    else:
        context_tuple = ()
        
    dist = model.next_word_distribution(context_tuple)
    
    # Sắp xếp các từ theo xác suất giảm dần
    sorted_words = sorted(dist.items(), key=lambda x: x[1], reverse=True)[:top_k]
    
    print(f"=== CONTEXT: '{context_str}' (Model n={model.n}) ===")
    for rank, (word, prob) in enumerate(sorted_words, 1):
        print(f"  Top {rank}: '{word}' | Probability: {prob:.6f}")
    print("-" * 50)

# 3. Chạy thử các câu kiểm tra
print("--- 1. Kiểm tra các câu dự đoán ĐÚNG ---")
inspect_next_word(bigram_model, "united states")
inspect_next_word(bigram_model, "in the")

print("\n--- 2. Kiểm tra các câu dự đoán SAI ---")
inspect_next_word(bigram_model, "machine learning")
inspect_next_word(bigram_model, "artificial intelligence")

--- 1. Kiểm tra các câu dự đoán ĐÚNG ---
=== CONTEXT: 'united states' (Model n=2) ===
  Top 1: '</s>' | Probability: 0.002152
  Top 2: 'and' | Probability: 0.001144
  Top 3: 'that' | Probability: 0.000518
  Top 4: 'of' | Probability: 0.000438
  Top 5: 'in' | Probability: 0.000353
--------------------------------------------------
=== CONTEXT: 'in the' (Model n=2) ===
  Top 1: 'first' | Probability: 0.007325
  Top 2: 'best' | Probability: 0.007228
  Top 3: 'most' | Probability: 0.007226
  Top 4: 'same' | Probability: 0.006791
  Top 5: 'world' | Probability: 0.005708
--------------------------------------------------

--- 2. Kiểm tra các câu dự đoán SAI ---
=== CONTEXT: 'machine learning' (Model n=2) ===
  Top 1: '</s>' | Probability: 0.000940
  Top 2: 'and' | Probability: 0.000866
  Top 3: 'to' | Probability: 0.000644
  Top 4: 'about' | Probability: 0.000456
  Top 5: 'how' | Probability: 0.000308
--------------------------------------------------
=== CONTEXT: 'artificial intelligence' (